# HydraY NNUE: 16 vs 8 output buckets (v9 pool, 160 superbatches)

Runtime -> **GPU A100** (High-RAM if offered), then "Run all". The data cell needs
about **75 GiB of free disk** (the three parts decompressed, plus Drive's cache).

### The test
Same net as in production, `(768x4kb_hm -> 1024)x2 -> pairwise -> 1024 -> 16 -> 1`,
with one variable: the number of material-count output buckets (separate l1 and l2
heads), selected by `bucket = (pieces - 2) / (32 / BUCKETS)`.

| arm | `BUCKETS` | bucket | file size |
|---|---|---|---|
| candidate | 16 | (pieces - 2) / 2 | 6,557,760 B |
| control | 8 | (pieces - 2) / 4 (as today) | 6,425,664 B |

**Run this notebook twice, once per arm, changing ONLY `BUCKETS`** in the cell
"configuration". Everything else is identical by construction: the v9 pool (three
globally shuffled parts, always read as one stream in the same order), 160
superbatches (about 8.2 epochs of the 1.94B records), StepLR 0.001 with a x0.1 drop
at superbatch 80, WDL 0.3, batch 16,384. bullet resumes a stage at the exact sample
where the previous one stopped, so stages and restarts do not change what a run sees.
The two runs can go in parallel on two sessions (different `NET_ID`s, no clash on Drive).

Why 16 and not 32: in the data, positions with an odd number of pieces are rare
(datagen skips those whose best move is a capture, so mid-exchange positions are
filtered out), yet qsearch evaluates them all the time. With 32 buckets they would
get heads of their own, trained on the least data; with 16 each head holds one even
and one odd piece count.

Trainer: `nnue/trainer/src/bin/trainer_deep.rs` on the `output-buckets-16` branch,
built with `HYDRAY_OUTPUT_BUCKETS=<BUCKETS>`. The engine on that branch loads both
layouts (an 8-bucket file gives each head the two slots it covers: same function).

### Steps
1. setup and configuration;
2. Rust, clone, architecture check, build;
3. data: the three parts decompressed to local disk once (each size checked);
4. training in stages of `STAGE` superbatches, each checkpoint copied to Drive;
5. final check and copy of the net to Drive.

If the session dies: run the cells up to "stage helpers", then the cell "RESTART".

### What to look at during the run
bullet prints the running loss for every superbatch. The two arms are the same
function family on the same data stream, so their running losses ARE comparable at
equal superbatch: write down the values at 20, 80 (just before the LR drop) and 160
for both arms. A lower loss does not prove more Elo (two capacity increases on v7
had a better loss and played worse): the verdict is the SPRT.

In [ ]:
# --- helper: any failed command stops the notebook, output streams live ---
import subprocess, os, re, glob

def sh(cmd):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end='', flush=True)
        out.append(line)
    if p.wait() != 0:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}')
    return ''.join(out)

gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader',
                     shell=True, capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'none')
assert gpu, 'no GPU: change the runtime type'
if 'A100' not in gpu:
    print('WARNING: not an A100. It still trains, only slower; check the disk below.')
sh('nproc; free -g | head -2; df -h /content | tail -1')

In [ ]:
# --- configuration ---
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'

BUCKETS  = 16           # 16 = candidate, 8 = control. The ONLY difference between the two runs.
TOTAL_SB = 160          # superbatches of 100M samples (6104 batches of 16,384)
STAGE    = 20           # checkpoint to Drive every STAGE superbatches (multiple of 10)
BRANCH   = 'output-buckets-16'
NET_ID   = f'hydray-v9-ob{BUCKETS}-{TOTAL_SB}sb'

# The v9 pool (MANIFEST.txt of nnue/data/v9pack). The record counts catch a truncated
# upload or a partial decompression, which otherwise end without an error.
EXPECTED_RECORDS = {
    'hydray_v9_part1.bin.zst': 647_094_461,
    'hydray_v9_part2.bin.zst': 647_094_461,
    'hydray_v9_part3.bin.zst': 647_094_463,
}

REPO    = '/content/hydray'
TRAINER = f'{REPO}/nnue/trainer'
LOCAL   = '/content/data'
NET_FILE_BYTES = {16: 6_557_760, 8: 6_425_664}[BUCKETS]
TRAINER_LINE   = f'trainer_deep: 1024 -> 16 -> 1, {BUCKETS} output buckets'

assert BUCKETS in (8, 16)
assert TOTAL_SB % STAGE == 0 and STAGE % 10 == 0, 'stage ends must be checkpoints (save_rate 10)'

def find(name):
    hits = glob.glob(f'{DRIVE}/**/{name}', recursive=True)
    assert hits, f'{name} not found on Drive'
    return hits[0]

PARTS = {name: find(name) for name in EXPECTED_RECORDS}
for name, path in PARTS.items():
    print(f'{name}: {os.path.getsize(path)/2**30:6.2f} GiB  {path}')
print(f'\narm: {BUCKETS} output buckets, net id {NET_ID}, expected net file {NET_FILE_BYTES:,} B')

In [ ]:
# --- Rust, zstd, clone, architecture check, build ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('apt-get -qq install -y zstd >/dev/null')
sh(f'rm -rf {REPO} && git clone --depth 1 --branch {BRANCH} https://github.com/ThomasGhione/HydraY {REPO}')
sh(f'cd {REPO} && git log --oneline -1')

# Not ceremonial: a past run trained for an hour an architecture other than the one
# believed, because the clone was wrong and a checkpoint name says nothing about its
# contents. The trainer, the oracle and the engine header must agree.
tr  = open(f'{TRAINER}/src/bin/trainer_deep.rs').read()
sd  = open(f'{TRAINER}/src/bin/sanity_deep.rs').read()
hpp = open(f'{REPO}/nnue/network_deep.hpp').read()
assert 'const HIDDEN_SIZE: usize = 1024;' in tr and 'const L1_SIZE: usize = 16;' in tr
assert 'option_env!("HYDRAY_OUTPUT_BUCKETS")' in tr, 'trainer_deep.rs does not read HYDRAY_OUTPUT_BUCKETS'
assert 'const OUTPUT_BUCKET_COUNTS: [usize; 2] = [16, 8];' in sd, 'sanity_deep.rs does not read both layouts'
assert 'inline constexpr int OUTPUT_BUCKETS = 16;' in hpp and 'LEGACY_OUTPUT_BUCKETS = 8;' in hpp
print(f'branch {BRANCH}: 1024 -> 16 -> 1 in trainer, oracle and engine; engine loads 16 and 8')

CARGO = f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH env CUDA_PATH=/usr/local/cuda HYDRAY_OUTPUT_BUCKETS={BUCKETS}'
sh(f'{CARGO} cargo build -r --bin trainer_deep --features cuda')
sh(f'{CARGO} cargo build -r --bin sanity_deep')

In [ ]:
# --- data: the three parts decompressed once, read as one stream ---
# One zstd process per part: the Drive mount caches what a process reads and frees it
# when the process exits, so the peak is the raw data plus one part's cache.
os.makedirs(LOCAL, exist_ok=True)
need = sum(32 * n for n in EXPECTED_RECORDS.values())
have = sum(os.path.getsize(f'{LOCAL}/{n[:-4]}') for n in EXPECTED_RECORDS if os.path.exists(f'{LOCAL}/{n[:-4]}'))
st = os.statvfs('/content'); free = st.f_bavail * st.f_frsize
print(f'free {free/2**30:.1f} GiB, still to write {(need - have)/2**30:.1f} GiB')
assert free > (need - have) + 14 * 2**30, 'not enough disk: use an A100 runtime (or one with more disk)'

DATA_FILES = []
for name, path in PARTS.items():
    raw = f'{LOCAL}/{name[:-4]}'                 # strip .zst
    want = 32 * EXPECTED_RECORDS[name]
    if not (os.path.exists(raw) and os.path.getsize(raw) == want):
        # --long=27: the parts are compressed with a 128 MiB window.
        sh(f'zstd -d -T0 --long=27 -f -c "{path}" > {raw}')
    got = os.path.getsize(raw)
    assert got == want, f'{name}: {got} bytes after decompression, expected {want}'
    print(f'{name}: {got // 32:,} records, size checked', flush=True)
    DATA_FILES.append(raw)

DATA = ','.join(DATA_FILES)
total = need // 32
print(f'\ntraining stream: {total/1e9:.3f}B records, {TOTAL_SB} superbatches = {TOTAL_SB*6104*16384/total:.2f} epochs')

In [ ]:
# --- stage helpers (RUN THIS ALSO WHEN RESTARTING) ---

def stage_cmd(end, start, resume_from):
    # STAGE_END must sit right in front of cargo: `STAGE_END=40 cd dir && cargo ...` gives
    # the variable to `cd` only, and the trainer then runs straight to TOTAL_SB. Same for
    # HYDRAY_OUTPUT_BUCKETS, which cargo also needs to keep the build of this arm.
    args = f'{DATA} {TOTAL_SB} {NET_ID}'
    if resume_from is not None:
        args += f' {start} checkpoints/{NET_ID}-{resume_from}'
    return f'{CARGO} STAGE_END={end} cargo run -r --bin trainer_deep --features cuda -- {args}'

def check_checkpoint(end):
    q = f'{TRAINER}/checkpoints/{NET_ID}-{end}/quantised.bin'
    assert os.path.exists(q), f'missing {q}'
    assert os.path.getsize(q) == NET_FILE_BYTES, \
        f'{q}: {os.path.getsize(q)} bytes, not the {BUCKETS}-bucket layout ({NET_FILE_BYTES})'

def save_to_drive(end):
    # Copies the checkpoint to Drive and CHECKS it arrived: the Drive mount writes
    # through a cache, so an unfinished upload would pass for a finished one.
    ck  = f'{TRAINER}/checkpoints/{NET_ID}-{end}'
    dst = f'{DRIVE}/{NET_ID}-{end}'
    sh(f'rm -rf "{dst}" && cp -r {ck} {DRIVE}/')
    size = lambda p: sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(p) for f in fs)
    assert os.path.isdir(dst) and size(dst) == size(ck), f'incomplete copy on Drive: {dst}'
    print(f'superbatch {end} on Drive ({size(dst)/2**20:.0f} MiB, checked)', flush=True)

def run_stages(done=0):
    assert done % STAGE == 0, f'{done} is not a stage boundary'
    prev = done if done else None
    for k in range(done // STAGE, TOTAL_SB // STAGE):
        end, start = (k + 1) * STAGE, k * STAGE + 1
        print(f'\n===== stage {k+1}/{TOTAL_SB//STAGE}: superbatches {start}-{end}, {BUCKETS} buckets =====', flush=True)
        out = sh(stage_cmd(end, start, prev))
        assert TRAINER_LINE in out, f'the trainer did not print "{TRAINER_LINE}": wrong build'
        check_checkpoint(end)
        save_to_drive(end)
        prev = end

In [ ]:
# --- training ---
# Do NOT run this cell when restarting: use the cell "RESTART" below.
run_stages(done=0)

In [ ]:
# --- final check, copy to Drive ---
final = f'{TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB}/quantised.bin'
check_checkpoint(TOTAL_SB)
sh(f'{TRAINER}/target/release/sanity_deep {final}')
sh(f'cp {final} "{DRIVE}/{NET_ID}.nnue" && cmp {final} "{DRIVE}/{NET_ID}.nnue"')
print(f'\nnet on Drive: {DRIVE}/{NET_ID}.nnue')
print('Sanity line 1 must say "' + str(BUCKETS) + ' output buckets"; startpos small, the mirrored')
print('pair (wN e4 / bN e5) identical, a queen up ~ +1000 or more.')

In [ ]:
# --- RESTART (ONLY if the session died mid-run) ---
#   1. run the cells from "helper" to "stage helpers" (the data cell reuses the parts
#      already on local disk, or decompresses them again in a new session);
#   2. do NOT run the training cell;
#   3. set RESTART = True and DONE = the last superbatch saved on Drive.
# With RESTART = False this cell does nothing, so "Run all" is safe.
RESTART = False
DONE    = 20

if not RESTART:
    print('restart off (RESTART = False): nothing to do')
else:
    ck = f'{DRIVE}/{NET_ID}-{DONE}'
    assert os.path.isdir(ck), f'checkpoint not found on Drive: {ck}'
    os.makedirs(f'{TRAINER}/checkpoints', exist_ok=True)
    sh(f'cp -r "{ck}" {TRAINER}/checkpoints/')
    check_checkpoint(DONE)
    run_stages(done=DONE)

## After both runs, in the engine (`output-buckets-16` branch)

Nets: `hydray-v9-ob16-160sb.nnue` and `hydray-v9-ob8-160sb.nnue`.

    git switch output-buckets-16 && make prod

**1. Reorder both nets for sparsity** (same function, faster l1; the production net
is reordered, a fresh one is not). Positions: any few thousand, e.g. from the pool,
which is already shuffled:

    g++ -std=c++23 -O2 -march=native nnue/tools/reorder.cpp nnue/nnue_deep.cpp -o /tmp/reorder
    zstd -dc --long=27 hydray_v9_part3.bin.zst | head -c 640000 > /tmp/sample.bin
    ./chess datagen-dump /tmp/sample.bin 20000 | sed -n 's/^[0-9]*: \([^|]*\) |.*/\1/p' > /tmp/positions.fen
    /tmp/reorder hydray-v9-ob16-160sb.nnue /tmp/positions.fen ob16.nnue
    /tmp/reorder hydray-v9-ob8-160sb.nnue  /tmp/positions.fen ob8.nnue

It ends with "check: ... identical evaluations" and keeps each file's layout.

**2. Checks**: `./chess nnue-selftest ob16.nnue` (and ob8) must print the same evals as
`sanity_deep` on the same file, and "nnue-selftest OK".

**3. Holdout loss per bucket** (optional, the only loss neither net trained on): the
first 20,000,400 records of `v8_std.t0.bin` are outside the pool.

    g++ -std=c++23 -O2 -march=native nnue/tools/holdoutloss.cpp nnue/nnue_deep.cpp -o /tmp/holdoutloss
    /tmp/holdoutloss ob16.nnue nnue/data/v8_std.t0.bin --count 20000000 --selfcheck 1000

The table always has 16 rows; for the 8-bucket net two consecutive rows share a head.
Running it on the checkpoints at 20, 40, ..., 160 of both arms shows whether the gap
between the arms is still opening at the end of the budget.

**4. SPRT** on ONE binary, the branch's, the two nets via `EvalFile`: 4+0.04,
Threads=1, bounds [0, 5].

    share/scripts/sprt.sh start ob16-vs-ob8 ./chess ./chess \
        NEW_OPTS="EvalFile=$PWD/ob16.nnue" BASE_OPTS="EvalFile=$PWD/ob8.nnue" CPUS=4-11 CONC=8

The 16-bucket layout costs no measurable speed (same 8-bucket net on dev vs branch:
-0.29%, CI [-0.60%, +0.11%], against -0.25% for the same binary run twice), so the SPRT
measures the heads, not the build.

Do not SPRT the checkpoint at superbatch 80: it comes before the LR drop and is not a
half-budget net (a pre-decay checkpoint once made HalfKA look like a loss).